# ChemBreak 16 — Hierarchical Adaptive MDP Safety Evaluation

**Environment:** Google Cloud Notebook Enterprise  
**Target:** ChemDFM  
**Fixed dataset:** 24 tasks  
**Pipeline:** Baseline → Learning Epoch 1 → Epoch 2 → Epoch 3 → Freeze → Optimized Evaluation → Results

CB16 keeps the familiar 24-task experiment flow and changes only the MDP value/state architecture. Run from top to bottom.


In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

if 'runner' in globals():
    try:
        runner.close()
    except Exception:
        pass
    del runner

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak16"
EXPERIMENT_REVISION = "CB16_HIER_MDP_MINI24_V1"
LIVE                = True
LIVE_PROGRESS       = True

content_root = Path('/content').resolve()
assert content_root.is_dir(), '/content unavailable — use Google Cloud Notebook Enterprise.'
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith('REPLACE_'), 'Set PROJECT_ID before live execution.'

storage_root = content_root / 'chembreak16_storage'
model_cache = storage_root / 'cache/huggingface/hub'
package_dir = storage_root / 'python_packages'
env_paths = {
    'HF_HOME': storage_root/'cache/huggingface',
    'TRANSFORMERS_CACHE': model_cache,
    'HUGGINGFACE_HUB_CACHE': model_cache,
    'TORCH_HOME': storage_root/'cache/torch',
    'XDG_CACHE_HOME': storage_root/'cache/xdg',
    'PIP_CACHE_DIR': storage_root/'cache/pip',
}
for key,path in env_paths.items():
    path=Path(path); path.mkdir(parents=True,exist_ok=True); os.environ[key]=str(path)
package_dir.mkdir(parents=True,exist_ok=True)
print('CB16 storage:', storage_root)


## C1 — Clone or update the GitHub repository

The notebook is the controller; CB16 source is loaded from the `chembreak16` folder in GitHub.


In [ ]:
checkout = content_root / 'chembreak16_repo'
def git(*args, cwd=None): subprocess.run(['git', *args], cwd=cwd, check=True)
if not (checkout/'.git').is_dir():
    git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(checkout))
else:
    git('fetch','origin',BRANCH,cwd=checkout); git('checkout',BRANCH,cwd=checkout); git('pull','--ff-only','origin',BRANCH,cwd=checkout)
PROJECT_DIR=(checkout/PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR/'pyproject.toml').is_file(), f'CB16 package not found at {PROJECT_DIR}. Push the chembreak16 folder to GitHub first.'
os.chdir(PROJECT_DIR)
print('Project:',PROJECT_DIR)


## C2 — Install the CB16 dependency stack

Dependencies and caches stay under `/content/chembreak16_storage`.


In [ ]:
compatibility_specs=['transformers==4.40.2','tokenizers==0.19.1','huggingface-hub==0.23.5','safetensors==0.4.5','accelerate==0.30.1','sentencepiece==0.2.0','einops==0.8.1']
marker=package_dir/'cb16_compatibility.json'; expected={'specifications':compatibility_specs}
installed=json.loads(marker.read_text()) if marker.exists() else None
if installed!=expected:
    print('Installing CB16 compatibility stack (first run only)...')
    subprocess.run([sys.executable,'-m','pip','install','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'--no-deps','--upgrade',*compatibility_specs],check=True)
    marker.write_text(json.dumps(expected,indent=2))
else:
    print('Compatibility stack already installed.')
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'google-auth>=2.35,<3','google-genai>=1.47,<2','pandas>=2.2,<3','numpy>=1.26,<3','PyYAML>=6,<7'],check=True)
site.addsitedir(str(package_dir)); sys.path.insert(0,str(package_dir)); sys.path.insert(0,str(PROJECT_DIR/'src')); importlib.invalidate_caches()
print('Dependency stack ready.')


## C3 — Verify the fixed 24-task CB16 panel

This verifies the CB16 manifest and lock against the 500-task source bank. No train/test/holdout split is created here.


In [ ]:
sys.path.insert(0,str(PROJECT_DIR/'src'))
for module_name in [name for name in list(sys.modules) if name == 'chembreak16' or name.startswith('chembreak16.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
from chembreak16.selection import verify_bundle
report=verify_bundle(PROJECT_DIR/'data/final_task_bank.csv',PROJECT_DIR/'data/CB16_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB16_mini24_lock_v1.json')
print(json.dumps(report,indent=2,sort_keys=True))


## C4 — Build the live runtime configuration

Only environment-specific paths, `LIVE`, and the experiment revision are changed. The fixed 24-task panel is unchanged.


In [ ]:
import yaml
base=yaml.safe_load((PROJECT_DIR/'configs/config.cb16.yaml').read_text())
base['run'].update({'project_root':str(PROJECT_DIR),'task_bank_path':str(PROJECT_DIR/'data/final_task_bank.csv'),'mini_manifest_path':str(PROJECT_DIR/'data/CB16_mini24_manifest_v1.csv'),'mini_lock_path':str(PROJECT_DIR/'data/CB16_mini24_lock_v1.json'),'output_root':str(storage_root/'runs'),'dry_run':not LIVE,'experiment_revision':EXPERIMENT_REVISION,'task_limit':None,'live_progress':LIVE_PROGRESS})
base['targets'][0]['cache_dir']=str(model_cache); base['targets'][0]['offload_folder']=str(storage_root/'offload/ChemDFM')
policy_dir=storage_root/'policies'/EXPERIMENT_REVISION
base['policy']['training_artifact_path']=str(policy_dir/'training_policy.json'); base['policy']['frozen_artifact_path']=str(policy_dir/'frozen_policy.json')
runtime_path=storage_root/f'runtime_{EXPERIMENT_REVISION}.yaml'; runtime_path.write_text(yaml.safe_dump(base,sort_keys=False))
if LIVE:
    os.environ['GOOGLE_CLOUD_PROJECT']=PROJECT_ID
    os.environ['CHEMBREAK_ENABLE_LIVE']='YES'
print('Runtime config:',runtime_path)
print('LIVE:',LIVE,'| LIVE_PROGRESS:',LIVE_PROGRESS,'| Project:',os.environ.get('GOOGLE_CLOUD_PROJECT','mock'))


## C5 — Preflight

Checks the CB16 panel, policy configuration, CUDA/GPU requirements, ChemDFM tokenizer compatibility, and structured-output role probes.


In [ ]:
from chembreak16.preflight import run_preflight
preflight=run_preflight(runtime_path,probe_tokenizer=LIVE,probe_roles=LIVE)
print(json.dumps(preflight,indent=2,sort_keys=True))
assert preflight['status']=='ok'


## C6 — Create the runner and load ChemDFM once

The loaded target is reused for Baseline, all three learning epochs, and Optimized evaluation.


In [ ]:
from chembreak16.runner import ChemBreak16Runner
runner=ChemBreak16Runner(runtime_path)
runner.load_target()
print('Runner ready.')
print('Planned episodes: 24 baseline + 72 learning + 24 optimized = 120')
print('Maximum target queries: 408 (usually lower because success stops an episode early)')


## C7 — Phase 1: Baseline

Each original benchmark prompt is sent once. No MDP strategy is selected and no Q-value is updated. With `LIVE_PROGRESS = True`, running Baseline ASR appears live.

The live progress line includes `running_ASR=...` after each completed task.


In [ ]:
baseline_summary=runner.run_baseline()
print(json.dumps(baseline_summary,indent=2,sort_keys=True))


## C8 — Phase 2: Learning (3 epochs)

The same 24 tasks are run in fresh conversations for three epochs. Base epsilon is **0.30 → 0.20 → 0.15**.

CB16 separates value into reusable global behavior (`Qg`), HC context (`Qhc`), HD context (`Qhd`), OT context (`Qot`), and a small task residual (`Qt`). Only components with learned support contribute to the weighted score. A truly unsupported state is shown as `mode=cold_start`. Adaptive epsilon and anti-repetition controls remain active.

Every adaptive turn prints mode, base→effective epsilon, all Q components, active components, support, repetition penalty, adjusted score, and blocked actions.


In [ ]:
learning_summary=runner.run_learning()
print(json.dumps(learning_summary,indent=2,sort_keys=True))


## C9 — Freeze the learned policy

Freezing makes the policy read-only. Optimized evaluation uses this exact policy with epsilon zero.


In [ ]:
frozen=runner.freeze_policy()
print(json.dumps(frozen,indent=2,sort_keys=True))


## C10 — Phase 3: Optimized evaluation

The same 24 task goals are evaluated in fresh conversations with the frozen hierarchical policy. `epsilon = 0` and Q-values are not updated. Live output shows where learned support is actually being reused.


In [ ]:
optimized_summary=runner.run_optimized()
print(json.dumps(optimized_summary,indent=2,sort_keys=True))


## C11 — Results

The headline comparison remains Baseline ASR vs Optimized ASR. `policy_diagnostics.csv` and `policy_support_summary.json` show whether the hierarchical value structure is producing real learned support instead of mostly zero-Q decisions.


In [ ]:
summary=runner.export_results()
print(json.dumps(summary,indent=2,sort_keys=True))
release_dir=storage_root/'runs'/EXPERIMENT_REVISION/'release'
print('Release directory:',release_dir)
print('Files:',[p.name for p in sorted(release_dir.iterdir())])


## C12 — Build the results ZIP and close the model

The ZIP contains release tables, checkpoint, runtime configuration, CB16 mini-set provenance, and training/frozen policy artifacts. Model weights and caches are excluded.


In [ ]:
import zipfile
runner.close()
run_dir=storage_root/'runs'/EXPERIMENT_REVISION
release_dir=run_dir/'release'
zip_path=content_root/f'{EXPERIMENT_REVISION}_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in release_dir.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(run_dir))
    checkpoint=run_dir/'state.sqlite3'
    if checkpoint.exists(): z.write(checkpoint,Path('checkpoint')/checkpoint.name)
    if runtime_path.exists(): z.write(runtime_path,Path('provenance')/runtime_path.name)
    for p in [PROJECT_DIR/'data/CB16_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB16_mini24_lock_v1.json']:
        z.write(p,Path('provenance')/p.name)
    for p in [policy_dir/'training_policy.json',policy_dir/'frozen_policy.json']:
        if p.exists(): z.write(p,Path('policies')/p.name)
print('Results ZIP:',zip_path)
